# NYC Taxi dataset anaylsis notebook

The objective is to perform **exploratory data analysis** on the `NYC Yellow Taxi` dataset to uncover patterns in taxi usage, create visualizations to interpret these patterns, and build a machine learning model to predict trip duration based on features like trip distance and pickup location.


## Step 1: Set Up the Environment

We start by initializing a PySpark session and connecting to the Connectors to access our datasets. Modify the variables at the top to have the correct dataset and project ID.

- **Note:** Replace `PROJECT_ID` with your actual ID from the URL (https://eu.dataplatform.ovh.net/dpe/#/{PROJECT_ID}/notebooks). Use default_dataset or your custom dataset name.

In [ ]:
import logging
from forepaas.dwh import connect
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, hour, dayofweek, unix_timestamp, avg, count, sum, when
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
 
# Set up variables (dataset, dataplant_url and YEAR, MONTH of the yellow taxi file
DATASET = "default_dataset"
PROJECT_ID = "PROJECT_ID"
YEAR = "2025"
MONTH = "01"
  
# Set up logging for debugging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
  
# Initialize SparkSession
try:
    spark = SparkSession.builder.appName("NYC_Yellow_Taxi_Analysis").getOrCreate()
    logging.info(f"Spark Version: {spark.version}")
except Exception as e:
    logging.error(f"Failed to initialize SparkSession: {e}")
    raise
  
# Connect to Lakehouse - default_dataset (default)
try:
    cn_prim = connect(f"dwh/{DATASET}/")
    logging.info(f"Connected to Lakehouse - Dataset: {DATASET}")
except Exception as e:
    logging.error(f"Failed to connect to Lakehouse - Dataset: {DATASET} | {e}")
    raise

### Explanation:

- **Why:** The `SparkSession` is the entry point for PySpark, allowing us to work with DataFrames and perform distributed computations. The `connect` function from `forepaas.dwh` links to the OVHcloud Connectors, which organizes datasets in the Lakehouse.
- **What:** We set up logging to track progress and catch errors. The `appName` helps identify the Spark job in the cluster.
- **Output:** Confirms the Spark version and successful connection to the Lakehouse dataset - default_dataset or other dataset if you create/use another one.

## Step 2: Load and Inspect Data

We load the `Yellow Taxi dataset` and the `Taxi Zone Lookup` table from the default_dataset and inspect their schemas.

In [ ]:
# Load datasets (Replace PROJECT_ID by yours, same for default_dataset if you create another one
yellow_df = cn_prim.query(f"SELECT * FROM db_{PROJECT_ID}_{DATASET}.{DATASET}.yellow_tripdata_{YEAR}_{MONTH}")
taxi_zones_df = cn_prim.query(f"SELECT LocationID, Borough, Zone FROM db_{PROJECT_ID}_{DATASET}.{DATASET}.taxi_zone_lookup")
 
# Cache DataFrames for performance
yellow_df.cache()
taxi_zones_df.cache()
 
# Verify data loading
logging.info(f"Yellow Taxi Records: {yellow_df.count()}")
logging.info(f"Taxi Zones Records: {taxi_zones_df.count()}")
 
# Inspect schemas
print("Yellow Taxi Schema:")
yellow_df.printSchema()
print("Taxi Zones Schema:")
taxi_zones_df.printSchema()

### Explanation:

- **Why:** The Connectors stores datasets in a structured format (Parquet for Yellow Taxi, CSV for Taxi Zones). Caching improves performance for repeated operations on large datasets.
- **What:** We load the datasets using SQL queries via `cn_prim.query`. The `count()` method verifies the number of records, and `printSchema()` shows the structure of the data.
- **Output:**
    - **Yellow Taxi Records:** ~3,475,226 rows.
    - **Taxi Zones Records:** 265 rows.
    - **Yellow Taxi Schema:** Includes fields like vendorid, tpep_pickup_datetime, trip_distance, fare_amount, pulocationid, and dolocationid.
    - **Taxi Zones Schema:** Includes LocationID, Borough, and Zone.

## Step 3: Clean the Data
We clean the `Yellow Taxi dataset` to remove invalid or incomplete records.

In [ ]:
# Clean Yellow Taxi DataFrame
yellow_df_clean = yellow_df.filter(
    (col("tpep_pickup_datetime").isNotNull()) &
    (col("tpep_dropoff_datetime").isNotNull()) &
    (col("passenger_count").isNotNull()) &
    (col("passenger_count") > 0) &
    (col("trip_distance") > 0) &
    (col("fare_amount") > 0)
)
 
# Verify cleaned data
logging.info(f"Cleaned Yellow Taxi Records: {yellow_df_clean.count()}")

### Explanation:

- **Why:** Cleaning removes records with missing or unrealistic values (zero passengers or negative fares) to ensure accurate analysis. Filtering out invalid trips (< 60 seconds, < 0.1 miles)
- **What:** We use filter with conditions to keep only valid records. The `col` function helps reference columns in PySpark.
- **Output:** ~2,816,835 records, indicating ~19% of records were removed due to invalid data.

## Step 4: Join with Taxi Zones
We join the `Yellow Taxi dataset` with the `Taxi Zone Lookup` table to add geographical context.

In [ ]:
# Join with taxi zones for pickup and dropoff locations
yellow_df_clean = yellow_df_clean.join(taxi_zones_df, yellow_df_clean.pulocationid == taxi_zones_df.LocationID, "left") \
    .withColumnRenamed("Borough", "pickup_borough") \
    .withColumnRenamed("Zone", "pickup_zone") \
    .drop("LocationID")
 
yellow_df_clean = yellow_df_clean.join(taxi_zones_df, yellow_df_clean.dolocationid == taxi_zones_df.LocationID, "left") \
    .withColumnRenamed("Borough", "dropoff_borough") \
    .withColumnRenamed("Zone", "dropoff_zone") \
    .drop("LocationID")
 
# Filter out invalid zones and boroughs
yellow_df_clean = yellow_df_clean.filter(
    (col("pickup_zone") != "Unknown") & (col("dropoff_zone") != "Unknown") &
    (col("pickup_borough") != "Unknown") & (col("pickup_borough") != "N/A") & (col("pickup_borough").isNotNull()) &
    (col("dropoff_borough") != "Unknown") & (col("dropoff_borough") != "N/A") & (col("dropoff_borough").isNotNull())
)
 
# Verify cleaned and joined data
logging.info(f"Cleaned Yellow Taxi Records after borough filtering: {yellow_df_clean.count()}")
yellow_df_clean.select("pulocationid", "pickup_borough", "pickup_zone", "dolocationid", "dropoff_borough", "dropoff_zone").show(5)

### Explanation:

- **Why:** Joining with taxi_zones_df maps `pulocationid` and `dolocationid` to human-readable boroughs and zones (Manhattan, Upper East Side).

- **What:** We perform two left joins to add pickup and dropoff locations, rename columns for clarity, and filter out records with "Unknown", "N/A", or null boroughs to ensure data quality.

- **Output:** Displays a sample of joined data:

| pulocationid | pickup_borough | pickup_zone          | dolocationid | dropoff_borough | dropoff_zone         |
|--------------|----------------|-----------------------|---------------|------------------|-----------------------|
| 237.0        | Manhattan       | Upper East Side South | 140.0         | Manhattan        | Lenox Hill East       |
| 239.0        | Manhattan       | Upper West Side South | 142.0         | Manhattan        | Lincoln Square East   |
| 140.0        | Manhattan       | Lenox Hill East       | 236.0         | Manhattan        | Upper East Side North |
| 68.0         | Manhattan       | East Chelsea          | 107.0         | Manhattan        | Gramercy              |
| 246.0        | Manhattan       | West Chelsea/Hudson Yards | 48.0     | Manhattan        | Clinton East          |

only showing top 5 rows

## Step 5: Feature Engineering and Enhanced Data Cleaning
We add derived features and apply additional cleaning to prepare the data for machine learning by removing invalid trips and capping outliers.

In [ ]:
# Add derived features
yellow_df_clean = yellow_df_clean.withColumn("pickup_hour", hour("tpep_pickup_datetime")) \
    .withColumn("day_of_week", dayofweek("tpep_pickup_datetime")) \
    .withColumn("trip_duration", unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) \
    .withColumn("total_revenue", col("fare_amount") + col("tip_amount") + col("congestion_surcharge") + col("airport_fee"))
 
# Inspect trip_duration before additional cleaning
yellow_df_clean.select("trip_duration").summary("count", "min", "max", "mean").show()
 
# Check for outliers
logging.info(f"Trips with duration > 1 hour: {yellow_df_clean.filter(col('trip_duration') > 3600).count()}")
logging.info(f"Trips with distance > 50 miles: {yellow_df_clean.filter(col('trip_distance') > 50).count()}")
 
# Filter out invalid trips and cap outliers
yellow_df_clean = yellow_df_clean.filter(
    (col("trip_duration") >= 60) &  # Minimum 1-minute trip duration
    (col("trip_distance") >= 0.1) &  # Minimum 0.1 miles
    (col("trip_distance").isNotNull()) &  # Remove null distances
    (col("pickup_hour").isNotNull()) &    # Remove null hours
    (col("tpep_pickup_datetime").isNotNull()) &  # Remove null pickup times
    (col("tpep_dropoff_datetime").isNotNull()) &   # Remove null dropoff times
    (col("passenger_count").isNotNull())  # Ensure passenger_count is not null
)
 
# Cap outliers
yellow_df_clean = yellow_df_clean.withColumn("trip_duration", when(col("trip_duration") > 3600, 3600).otherwise(col("trip_duration"))) \
    .withColumn("trip_distance", when(col("trip_distance") > 50, 50).otherwise(col("trip_distance")))
 
# Verify cleaned data after filtering and capping
logging.info(f"Yellow Taxi Records after duration filtering and outlier capping: {yellow_df_clean.count()}")
yellow_df_clean.select("trip_duration").summary("count", "min", "max", "mean").show()

### Explanation:

- **Why:**

    - **Data Quality for ML:** Machine learning models require clean, consistent data to produce reliable predictions. Negative or extremely high trip_duration values (-3362 or 337,579 seconds) and unrealistic trip_distance values (>50 miles) indicate data errors or outliers that can skew model training. Filtering short trips (<1 minute) and capping outliers (durations >1 hour, distances >50 miles) ensures the data is realistic for NYC taxi trips.

    - **Feature Engineering:** Derived features like `pickup_hour`, `day_of_week`, `trip_duration`, and `total_revenue` enable temporal and financial analysis and serve as input features for ML models.

- **What:**

    - We inspect `trip_duration` to identify outliers (28,432 trips >1 hour, 43 trips >50 miles).

    - We add features using `withColumn` for `pickup_hour`, `day_of_week`, `trip_duration`, and `total_revenue`.

    - We filter out trips with durations <60 seconds or distances <0.1 miles, and ensure non-null values.

    - We cap `trip_duration` at 3600 seconds (1 hour) and `trip_distance` at 50 miles to handle outliers.

- **Output:**

    - Initial trip_duration summary: ~2,792,399 records, min -3362, max 337,579, mean ~901.62 seconds.

    - After filtering and capping: ~2,781,003 records, min 60, max 3600, mean ~867.17 seconds.

    - This confirms the removal of ~1.2% of records and a more realistic dataset for ML.

## Step 6: Exploratory Data Analysis (EDA)
We perform EDA to uncover patterns in taxi usage.

### Analysis 1: Trips by Hour

In [ ]:
# Group by pickup_hour and count trips
yellow_hourly_trips = yellow_df_clean.groupBy("pickup_hour") \
    .agg(count("*").alias("num_trips")) \
    .orderBy("pickup_hour")
yellow_hourly_trips_pd = yellow_hourly_trips.toPandas()
 
# Visualize
plt.figure(figsize=(10, 6))
plt.bar(yellow_hourly_trips_pd["pickup_hour"], yellow_hourly_trips_pd["num_trips"], color="skyblue")
plt.xlabel("Hour of Day")
plt.ylabel("Number of Trips")
plt.title("Yellow Taxi Trips per Hour (January 2025)")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

#### Explanation:

- **Why:** This analysis shows how taxi demand varies by hour, helping identify peak travel times (rush hours).
- **What:** We group by `pickup_hour`, `count trips`, and convert to a Pandas DataFrame for visualization with Matplotlib.
- **Result:** The bar plot likely shows peaks during morning (7–9 AM) and evening (5–7 PM) rush hours, reflecting commuting patterns in NYC. Nighttime hours (2–4 AM) have fewer trips due to lower demand.

### Analysis 2: Average Trip Duration by Day of Week

In [ ]:
# Group by day_of_week and calculate average trip duration
yellow_duration_by_day = yellow_df_clean.groupBy("day_of_week") \
    .agg(avg("trip_duration").alias("avg_duration_seconds")) \
    .orderBy("day_of_week")
yellow_duration_by_day_pd = yellow_duration_by_day.toPandas()
yellow_duration_by_day_pd["avg_duration_minutes"] = yellow_duration_by_day_pd["avg_duration_seconds"] / 60
 
# Visualize
plt.figure(figsize=(10, 6))
plt.plot(yellow_duration_by_day_pd["day_of_week"], yellow_duration_by_day_pd["avg_duration_minutes"], marker='o', color='coral')
plt.xlabel("Day of Week (1=Sunday, 7=Saturday)")
plt.ylabel("Average Trip Duration (Minutes)")
plt.title("Average Yellow Taxi Trip Duration by Day of Week (January 2025)")
plt.grid(True)
plt.show()

#### Explanation:

- **Why:** Trip duration varies by day due to differences in traffic or trip purpose.
- **What:** We calculate the average `trip_duration` per day of the week and convert seconds to minutes for readability.
- **Result:** The line plot may show longer trips on weekdays and less traffic the weekends.

### Analysis 3: Revenue by Borough

In [ ]:
# Group by pickup_borough and calculate average revenue
yellow_revenue_by_borough = yellow_df_clean.groupBy("pickup_borough") \
    .agg(avg("total_revenue").alias("avg_revenue"), count("*").alias("num_trips")) \
    .orderBy("avg_revenue", ascending=False)
yellow_revenue_by_borough_pd = yellow_revenue_by_borough.toPandas()
 
# Visualize
plt.figure(figsize=(10, 6))
sns.barplot(data=yellow_revenue_by_borough_pd, x="avg_revenue", y="pickup_borough", hue="pickup_borough", palette="Blues_d")
plt.xlabel("Average Revenue ($)")
plt.ylabel("Pickup Borough")
plt.title("Average Yellow Taxi Revenue by Pickup Borough (January 2025)")
plt.show()

#### Explanation:

- **Why:** Revenue analysis helps identify which boroughs generate the most income, useful for taxi operators or urban planners.
- **What:** We calculate the average `total_revenue` per borough and visualize it with Seaborn for a cleaner presentation.
- **Result:** Boroughs like Queens and EWR (Newark Airport) likely show higher average revenue due to longer trips (airport rides). In analysis 5, we will check what is the boroughs with the most trips.

### Analysis 4: Tipping Behavior for Credit Card Payments

In [ ]:
# Filter for credit card payments (payment_type = 1)
yellow_credit_tips = yellow_df_clean.filter(col("payment_type") == 1) \
    .groupBy("pickup_borough") \
    .agg(avg("tip_amount").alias("avg_tip"), count("*").alias("num_tipped_trips")) \
    .orderBy("avg_tip", ascending=False)
yellow_credit_tips_pd = yellow_credit_tips.toPandas()
 
# Visualize
plt.figure(figsize=(10, 6))
sns.barplot(data=yellow_credit_tips_pd, x="avg_tip", y="pickup_borough", hue="pickup_borough", palette="Greens_d")
plt.xlabel("Average Tip ($)")
plt.ylabel("Pickup Borough")
plt.title("Average Tip Amount for Credit Card Payments by Pickup Borough (January 2025)")
plt.show()

#### Explanation:

- **Why:** Tipping behavior varies by borough and is only recorded for credit card payments (payment_type = 1).
- **What:** We filter for credit card payments, calculate average tips per borough, and visualize the results.
-  **Result:** EWR and Queens likely show higher tips due to longer trips (airport rides). Bronx lower tips reflect shorter urban trips where tipping is less common.

### Analysis 5: Identifying the Zone with the Most Trips

In [ ]:
# Analysis 5: Trips by Pickup Zone
yellow_zone_trips = yellow_df_clean.groupBy("pickup_zone") \
    .agg(count("*").alias("num_trips")) \
    .orderBy("num_trips", ascending=False)
 
# Limit to top 10 zones for visualization
yellow_zone_trips_top10 = yellow_zone_trips.limit(10)
yellow_zone_trips_top10_pd = yellow_zone_trips_top10.toPandas()
 
# Visualize
plt.figure(figsize=(12, 6))
sns.barplot(data=yellow_zone_trips_top10_pd, x="num_trips", y="pickup_zone", hue="pickup_zone", palette="Purples_d")
plt.xlabel("Number of Trips")
plt.ylabel("Pickup Zone")
plt.title("Top 10 Yellow Taxi Pickup Zones by Number of Trips (January 2025)")
plt.show()
 
# Log the top zone
top_zone = yellow_zone_trips_top10_pd.iloc[0]["pickup_zone"]
top_zone_trips = yellow_zone_trips_top10_pd.iloc[0]["num_trips"]
logging.info(f"Zone with the most trips: {top_zone} with {top_zone_trips} trips")

#### Explanation
- **Why:** Grouping by pickup_zone and counting trips reveals which specific areas (Upper East Side, Midtown) have the highest taxi demand. Limiting to the top 10 zones keeps the visualization manageable, as there are 265 zones in the taxi_zone_lookup table.
- **What:**
    - The `groupBy("pickup_zone")` operation aggregates trips by pickup zone.
    - The `count("*").alias("num_trips")` counts the number of trips per zone.
    - The `orderBy("num_trips", ascending=False)` sorts zones by trip count in descending order.
    - The `limit(10)` selects the top 10 zones to avoid cluttering the visualization.
    - The data is converted to a Pandas DataFrame `(toPandas())` for visualization with Seaborn’s barplot, which is ideal for categorical data.
    - The `Purples_d` palette provides a visually appealing gradient.
    - Logging the top zone provides a clear summary of the result.
- **Result:** 
    - **Expected Top Zones:** Based on historical NYC taxi data, zones like Upper East Side, Midtown, Times Square often have the most trips due to their status as residential, business, or tourist hubs. For example:
    - **Upper East Side:** A dense residential area with high-income residents who frequently use taxis.
    - **Midtown:** Commercial and tourist areas with heavy foot traffic.
    - **Airport Zones (JFK or LaGuardia):** May appear if airport trips are common, though they typically have fewer trips but higher revenue.

### Step 7: Machine Learning Model to Predict Trip Duration
In this step, we build and compare three machine learning models - Linear Regression, Random Forest Regressor, and Gradient Boosting Trees (GBT) - to predict the trip_duration of Yellow Taxi trips using features like `trip_distance`, `pickup_hour`, `day_of_week`, and `pickup_borough`. This analysis is valuable for taxi operators to optimize scheduling, estimate fares, or improve operational efficiency. We preprocess the data, train the models, evaluate their performance using Root Mean Squared Error (RMSE) and R-squared metrics, and interpret feature importance to understand which factors most influence trip duration.

**Note:** This step may take a few minutes to execute, depending on the number of Data Processing Units (DPUs) allocated to your Jupyter notebook.

In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
import logging
 
# Step 7.1: Prepare Features
# Drop pickup_borough_index if it already exists to avoid conflicts
if "pickup_borough_index" in yellow_df_clean.columns:
    yellow_df_clean = yellow_df_clean.drop("pickup_borough_index")
    logging.info("Dropped existing pickup_borough_index column to avoid conflict.")
 
# Encode pickup_borough as a numerical feature
indexer = StringIndexer(inputCol="pickup_borough", outputCol="pickup_borough_index")
yellow_df_clean = indexer.fit(yellow_df_clean).transform(yellow_df_clean)
 
# Assemble initial feature vector (excluding pickup_borough_index for scaling)
assembler_initial = VectorAssembler(inputCols=["trip_distance", "pickup_hour", "day_of_week"], outputCol="features_initial")
data_initial = assembler_initial.transform(yellow_df_clean)
 
# Standardize features
scaler = StandardScaler(inputCol="features_initial", outputCol="features_scaled", withStd=True, withMean=True)
scaler_model = scaler.fit(data_initial)
data_scaled = scaler_model.transform(data_initial).drop("features_initial")
 
# Combine scaled features with pickup_borough_index
assembler_final = VectorAssembler(inputCols=["features_scaled", "pickup_borough_index"], outputCol="features")
data = assembler_final.transform(data_scaled).select("features", "trip_duration")
 
# Step 7.2: Split Data into Training and Test Sets
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)
logging.info(f"Training data records: {train_data.count()}")
logging.info(f"Test data records: {test_data.count()}")
 
# Step 7.3: Train and Compare Models
# Linear Regression
lr = LinearRegression(featuresCol="features", labelCol="trip_duration")
lr_model = lr.fit(train_data)
lr_predictions = lr_model.transform(test_data)
 
# Random Forest Regressor
rf = RandomForestRegressor(featuresCol="features", labelCol="trip_duration", numTrees=100, seed=42)
rf_model = rf.fit(train_data)
rf_predictions = rf_model.transform(test_data)
 
# Gradient Boosting Trees
gbt = GBTRegressor(featuresCol="features", labelCol="trip_duration", maxIter=50, seed=42)
gbt_model = gbt.fit(train_data)
gbt_predictions = gbt_model.transform(test_data)
 
# Step 7.4: Evaluate Models
evaluator = RegressionEvaluator(labelCol="trip_duration", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="trip_duration", predictionCol="prediction", metricName="r2")
 
# Evaluate Linear Regression
lr_rmse = evaluator.evaluate(lr_predictions)
lr_r2 = evaluator_r2.evaluate(lr_predictions)
logging.info(f"Linear Regression - Root Mean Squared Error (RMSE): {lr_rmse}")
logging.info(f"Linear Regression - R-squared: {lr_r2}")
 
# Evaluate Random Forest
rf_rmse = evaluator.evaluate(rf_predictions)
rf_r2 = evaluator_r2.evaluate(rf_predictions)
logging.info(f"RandomForest - Root Mean Squared Error (RMSE): {rf_rmse}")
logging.info(f"RandomForest - R-squared: {rf_r2}")
 
# Evaluate Gradient Boosting
gbt_rmse = evaluator.evaluate(gbt_predictions)
gbt_r2 = evaluator_r2.evaluate(gbt_predictions)
logging.info(f"GBT - Root Mean Squared Error (RMSE): {gbt_rmse}")
logging.info(f"GBT - R-squared: {gbt_r2}")
 
# Show sample predictions for all models
logging.info("Linear Regression Predictions:")
lr_predictions.select("features", "trip_duration", "prediction").show(5)
logging.info("RandomForest Predictions:")
rf_predictions.select("features", "trip_duration", "prediction").show(5)
logging.info("GBT Predictions:")
gbt_predictions.select("features", "trip_duration", "prediction").show(5)
 
# Check feature importance for RandomForest
logging.info(f"RandomForest Feature Importances: {rf_model.featureImportances}")